# step07_noremoval - productivity calibration WITHOUT NPQ removal (P45_06)

**Comparison variant (requested by Pierre).** IDENTICAL to step07 (same labels, sources and inference) EXCEPT
it KEEPS the daytime near-surface NPQ-flagged points in the fit (EXCLUDE_NPQ = False). Outputs go to
`_audit/productivity_noNPQ` and `L3/productivity_noNPQ`, so the main step07 results are not overwritten. This is
a sensitivity check, not the recommended product: surface daytime fluorescence is light-suppressed and reads low
relative to true Chl-a, so including those points pulls the fit. Compare gain, R2 and RMSE against step07.

Calibrates the CTD's biological/productivity sensors against discrete bottle samples, the same depth-matched
regression we used for the SUNA nitrate, generalized so it handles more than one parameter from one notebook:

- **Chlorophyll-a**: the WET Labs ECO fluorometer (`flECO-AFL`) vs extracted (fluorometric) chlorophyll. ACTIVE.
- **Dissolved oxygen**: the SBE 43 (`sbeox0Mm/L`) vs Winkler titration. BUILT AND READY, dormant until the
  Winkler file exists (P45-08). When that file is dropped in, this target activates with no code change.

**Why a sensor needs a discrete calibration.** A factory fluorometer calibration is only a rough default and is
routinely off by a factor of 2 to 5, and an SBE 43 oxygen sensor drifts between calibrations, so both are brought
onto an absolute scale against bottle references (the fluorometer against extracted chl, the oxygen against
Winkler, which is the gold standard: Carpenter 1965; Langdon, GO-SHIP 2010). The correction is the same two-term
regression as the nitrate: `sensor = a + b * bottle`, inverted to `corrected = (sensor - a) / b`.

**Matching.** Discrete samples are matched to the sensor by DEPTH (median of the downcast within `DEPTH_WIN` m
of the sample depth), because bottle firing times are cast-level, not exact. The station-to-cast link is read
from the bottle pairing audit, so the chl/O2 files do not need a `cast_id` column.

**Non-photochemical quenching (chlorophyll only), and why it is handled explicitly.** In daytime surface water,
sunlight suppresses chlorophyll fluorescence, so the fluorometer reads low relative to the true chl (Roesler et
al., 2017). On P45_06 the surface (5 m) fluorometer read only about 20% of the extracted chl on every afternoon
cast, a clear quenching signature. In this NO-REMOVAL variant those points are still FLAGGED (and marked on the scatter as "NPQ-flagged (kept)")
but are RETAINED in the fit, so you can see how much they bias the slope relative to the NPQ-removed step07.
The recommended, quenching-corrected calibration remains the one in step07.

**What the sensor is, and the honest label.** The ECO puck does not measure chlorophyll-a; it measures
in-vivo chlorophyll FLUORESCENCE (blue ~470 nm excitation, red ~695 nm emission), a proxy whose ratio to true
Chl-a varies with community composition, photoacclimation, nutrient status and quenching (Lorenzen 1966; Kiefer
1973; Falkowski and Kiefer 1985). So the raw channel is reported as FLUORESCENCE (factory chl units) and the
bottle-calibrated product as "Chl-a (fluorescence-derived)", not as a direct Chl-a measurement. The discrete
reference is extracted fluorometric Chl-a with acidification for phaeopigments (Holm-Hansen 1965; Welschmeyer
1994), one step below HPLC in the accuracy hierarchy (HPLC > extracted fluorometric > in-situ fluorescence).
Calibrating the fluorescence against these bottles is the standard way to turn the proxy into a quantitative
Chl-a estimate (Roesler et al. 2017; Schmechtig et al. 2018, BGC-Argo).

Sources: Lorenzen (1966), *Deep-Sea Res.*; Holm-Hansen et al. (1965), *ICES J. Mar. Sci.*; Welschmeyer (1994),
*L&O*; Kiefer (1973), *Mar. Biol.*; Cullen (1982), *Can. J. Fish. Aquat. Sci.*; Falkowski and Kiefer (1985),
*J. Plankton Res.*; Roesler et al. (2017), *L&O: Methods* (WET Labs ECO factory bias and NPQ); Schmechtig et al.
(2018), BGC-Argo chlorophyll processing; Langdon (2010), GO-SHIP oxygen; Carpenter (1965), Winkler; Sea-Bird
SBE 43 and WET Labs ECO documentation.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib
INTERACTIVE_ZOOM = True
try: get_ipython().run_line_magic("matplotlib","widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM=False
    try: get_ipython().run_line_magic("matplotlib","inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

CRUISE_ID  = "P45_06"
CTD_ROOT   = Path(r"C:\Projects\ctd_pipeline")
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
# CTD cast profiles: the 1 m downcast if present, else raw per-cast .cnv. cast_id = filename stem minus suffix.
CTD_DIR    = CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down"
CTD_SUFFIX = "_1m_down.cnv"
if not CTD_DIR.exists():
    CTD_DIR = CRUISE_DIR / "L1" / "CTD"; CTD_SUFFIX = ".cnv"
META       = CRUISE_DIR / "metadata"
PAIR_AUDIT = META / "bottle_nitrate" / f"{CRUISE_ID}_bottle_pairing_audit.csv"   # station -> cast
OUT_DIR    = CRUISE_DIR / "_audit" / "productivity_noNPQ"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L3_DIR     = CRUISE_DIR / "L3" / "productivity_noNPQ";     L3_DIR.mkdir(parents=True, exist_ok=True)

DEPTH_WIN   = 2.0      # m around each sample depth to take the sensor median (downcast)
MIN_IN_WIN  = 3        # need this many sensor scans to use a sample
EXCLUDE_NPQ = False    # NO-REMOVAL VARIANT: KEEP the NPQ-flagged near-surface points in the fit
NPQ_DEPTH   = 10.0     # m; samples at/above this on daytime casts are NPQ-flagged (chl only)
DAY_START, DAY_END = 7, 17   # local daylight hours for the NPQ test

# One entry per productivity parameter. Add more here; each activates when its discrete file exists.
TARGETS = {
    "chl": dict(label="Chlorophyll (fluorescence)", units="mg/m3",
                sensor_label="in-situ fluorescence (factory chl units)",
                discrete_label="extracted Chl-a", product_label="Chl-a (fluorescence-derived)",
                sensor_cols=["flECO-AFL","flECO-AFL1","wetCDOM","flC"],
                discrete=META/"chla"/f"{CRUISE_ID}_chla.csv",
                file_glob=["*chla*.csv","*chl*.csv"],
                value_col_candidates=["chl","chl_a","chla","chlorophyll","chlorophyll_a"],
                is_chl=True),
    "oxygen": dict(label="Dissolved oxygen", units="umol/L",
                sensor_label="SBE 43 oxygen", discrete_label="Winkler O2",
                product_label="O2 (Winkler-calibrated)",
                sensor_cols=["sbeox0Mm/L","sbeox0Mm/Kg","sbox0Mm/Kg","oxsolMm/L"],
                discrete=META/"winkler_oxygen"/f"{CRUISE_ID}_winkler.csv",
                file_glob=["*winkler*.csv","*oxygen*.csv","*o2*.csv"],
                value_col_candidates=["oxygen_uM","oxygen","winkler_uM","o2_umol_l","do_umol_l","oxygen_umol_kg"],
                is_chl=False),
}
print("cruise:", CRUISE_ID, "| CTD:", CTD_DIR, "| out:", L3_DIR)

In [ ]:
# ------------------------------------------------------------------ readers
def parse_cnv_downcast(path):
    """Return (names, pressure_down, DataFrame_down) for a cast's downcast (surface to pressure max)."""
    lines=Path(path).read_text(encoding="latin-1",errors="replace").splitlines(); names=[]; iend=None
    for i,ln in enumerate(lines):
        if ln.startswith("# name"): names.append(ln.split("=",1)[1].split(":")[0].strip())
        if ln.strip().upper().startswith("*END*"): iend=i+1; break
    rows=[]
    for ln in lines[iend:]:
        p=ln.split()
        if len(p)>=len(names):
            try: rows.append([float(x) for x in p[:len(names)]])
            except ValueError: pass
    d=np.array(rows,float); d[(d<=-8e-29)&(d>=-1.2e-28)]=np.nan
    df=pd.DataFrame(d,columns=names)
    pcol=next((c for c in names if re.search(r"pr[dD]M|prM|pressure|depS",c)),None)
    if pcol is None: return names, df
    imax=int(np.nanargmax(df[pcol].to_numpy()))
    return names, df.iloc[:imax+1].reset_index(drop=True), pcol

def load_station_cast():
    """station -> cast_id, from the bottle pairing audit (reused for all discrete productivity samples)."""
    if not PAIR_AUDIT.exists(): return {}
    a=pd.read_csv(PAIR_AUDIT); cols={c.lower():c for c in a.columns}
    s=cols.get("station"); c=cols.get("matched_cast") or cols.get("cast_id")
    return {str(r[s]).strip():str(r[c]).strip() for _,r in a.iterrows()} if (s and c) else {}

def find_ctd_files():
    out={}
    for p in sorted(CTD_DIR.glob(f"*{CTD_SUFFIX}")):
        out[p.name[:-len(CTD_SUFFIX)]]=p
    return out

def _resolve_discrete(cfg):
    """Find the discrete file tolerant of P45-06 vs P45_06 and name suffixes (uses file_glob)."""
    f=cfg["discrete"]
    if f.exists(): return f
    folder=f.parent
    if not folder.exists(): return None
    for pat in cfg.get("file_glob",["*.csv"]):
        hits=sorted(folder.glob(pat))+sorted(folder.glob(pat.upper()))
        if hits: return hits[0]
    return None

def load_discrete(cfg, s2c):
    """Read a discrete file, resolve cast_id (from its own column or station map), pick the value column."""
    f=_resolve_discrete(cfg)
    if f is None: return None, None
    if f.name != cfg["discrete"].name: print(f"  using discrete file: {f.name}")
    d=pd.read_csv(f); d.columns=[c.strip() for c in d.columns]
    low={c.lower():c for c in d.columns}
    vcol=next((low[v] for v in cfg["value_col_candidates"] if v in low), None)
    if vcol is None: raise ValueError(f"{f.name}: no value column among {cfg['value_col_candidates']}; has {list(d.columns)}")
    dcol=next((low[v] for v in ["depth","depth_m","deps","pressure"] if v in low), None)
    scol=low.get("station"); ccol=low.get("cast_id") or low.get("cast")
    out=pd.DataFrame(dict(value=pd.to_numeric(d[vcol],errors="coerce"),
                          depth_m=pd.to_numeric(d[dcol],errors="coerce")))
    out["station"]=d[scol].astype(str).str.strip() if scol else ""
    cid = d[ccol].astype(str).str.strip() if ccol else pd.Series([""]*len(d))
    cid = cid.where(cid.str.startswith("P45"), out["station"].map(s2c))
    out["cast_id"]=cid
    if "date_time" in low: out["time"]=pd.to_datetime(d[low["date_time"]],errors="coerce")
    elif "time_utc" in low: out["time"]=pd.to_datetime(d[low["time_utc"]],errors="coerce")
    else: out["time"]=pd.NaT
    return out.dropna(subset=["value","depth_m","cast_id"]), vcol

In [ ]:
# ------------------------------------------------------------------ match, flag, fit per target
_T975={1:12.71,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,8:2.306,9:2.262,10:2.228}
S2C=load_station_cast(); CTDF=find_ctd_files()
print(f"{len(CTDF)} CTD cast files in {CTD_DIR.name}; station->cast map: {len(S2C)} entries\n")

def sensor_at(names, df, pcol, scol, z):
    if scol not in df.columns: return np.nan,0
    m=df[(df[pcol]>=z-DEPTH_WIN)&(df[pcol]<=z+DEPTH_WIN)]
    v=m[scol].to_numpy(); v=v[np.isfinite(v)]
    return (float(np.median(v)),len(v)) if len(v) else (np.nan,0)

RESULTS={}
for key,cfg in TARGETS.items():
    disc,vcol=load_discrete(cfg,S2C)
    print(f"=== {cfg['label']} ({key}) ===")
    if disc is None:
        print(f"  no discrete file yet ({cfg['discrete'].name}); target READY, skipped.\n"); continue
    # average replicates per (cast, depth)
    disc=disc.groupby(["cast_id","station","depth_m"],as_index=False).agg(
        value=("value","mean"), n_rep=("value","size"), time=("time","first"))
    rows=[]
    for _,r in disc.iterrows():
        cid=r["cast_id"]; path=CTDF.get(cid)
        if path is None:
            rows.append(dict(cast=cid,station=r["station"],depth_m=r["depth_m"],bottle=r["value"],
                             sensor=np.nan,n=0,used=False,npq=False,note="no CTD cast file")); continue
        names,ddf,pcol=parse_cnv_downcast(path)
        scol=next((c for c in cfg["sensor_cols"] if c in names),None)
        sv,n=sensor_at(names,ddf,pcol,scol,float(r["depth_m"])) if scol else (np.nan,0)
        hour=r["time"].hour if pd.notna(r["time"]) else 12
        npq = bool(cfg["is_chl"] and r["depth_m"]<=NPQ_DEPTH and DAY_START<=hour<DAY_END)
        used = (n>=MIN_IN_WIN) and not (npq and EXCLUDE_NPQ)
        rows.append(dict(cast=cid,station=r["station"],depth_m=r["depth_m"],bottle=r["value"],
                         sensor=sv,n=n,used=used,npq=npq,
                         note=("NPQ-flagged" if npq else ("<%d scans"%MIN_IN_WIN if n<MIN_IN_WIN else ""))))
    cal=pd.DataFrame(rows); fit=cal[cal["used"]].dropna(subset=["sensor"])
    with pd.option_context("display.width",200):
        print(cal[["cast","station","depth_m","bottle","sensor","n","npq","used","note"]].round(3).to_string(index=False))
    off=gn=r2=np.nan; stats={}
    if len(fit)>=2 and np.ptp(fit["bottle"])>0:
        b=fit["bottle"].to_numpy(); s=fit["sensor"].to_numpy()
        (gn,off),cov=np.polyfit(b,s,1,cov=True)
        pred=off+gn*b; r2=1-np.sum((s-pred)**2)/np.sum((s-s.mean())**2)
        corr_at=(s-off)/gn; dof=max(len(fit)-2,1); tc=_T975.get(dof,1.96)
        se_g=float(np.sqrt(cov[0,0])); se_o=float(np.sqrt(cov[1,1]))
        stats=dict(n=len(fit),offset=off,gain=gn,R2=r2,pearson_r=float(np.corrcoef(b,s)[0,1]),
                   gain_lo=gn-tc*se_g,gain_hi=gn+tc*se_g,off_lo=off-tc*se_o,off_hi=off+tc*se_o,
                   rmse_raw=float(np.sqrt(np.mean((s-b)**2))),
                   rmse_corr=float(np.sqrt(np.mean((corr_at-b)**2))),
                   factory_factor=1.0/gn if gn else np.nan, n_npq=int(cal["npq"].sum()))
        print(f"\n  calibration:  sensor = {off:+.4f} + {gn:.4f} * {key}   ->   {key}_cal = (sensor - ({off:+.4f})) / {gn:.4f}")
        print(f"  gain {gn:.3f} (95% CI {stats['gain_lo']:.3f} to {stats['gain_hi']:.3f}) | offset {off:+.3f} | R2 {r2:.3f} | n {len(fit)}")
        print(f"  RMSE vs bottle: raw {stats['rmse_raw']:.3f} -> corrected {stats['rmse_corr']:.3f} {cfg['units']}")
        if cfg["is_chl"]: print(f"  factory->true multiplier ~{stats['factory_factor']:.2f} | NPQ-flagged points KEPT in fit: {stats['n_npq']}")
    else:
        print("\n  not enough usable points to fit.")
    RESULTS[key]=dict(cfg=cfg,cal=cal,fit=fit,offset=off,gain=gn,r2=r2,stats=stats)
    cal.to_csv(OUT_DIR/f"{CRUISE_ID}__{key}_calibration.csv",index=False,encoding="utf-8-sig")
    print()

## 4b. Inference from the results

In [ ]:
# ------------------------------------------------------------------ inference from the results
# Reads the fitted calibration and states what it means scientifically (not just the numbers).
def infer(key):
    R=RESULTS.get(key)
    if R is None or not np.isfinite(R["offset"]):
        print(f"{key}: no calibration (discrete file absent or too few usable points).\n"); return
    cfg=R["cfg"]; st=R["stats"]; gn=R["gain"]; off=R["offset"]; r2=R["r2"]
    ci_excl1 = (st["gain_lo"] > 1.0) or (st["gain_hi"] < 1.0)
    print(f"=== Inference: {cfg['label']} ===")
    if cfg["is_chl"]:
        ff=st["factory_factor"]; direction="over-reads" if gn>1 else "under-reads"; pct=abs(gn-1)*100
        print(f"- The ECO puck measures in-vivo chlorophyll FLUORESCENCE, a proxy for Chl-a, not Chl-a itself.")
        print(f"- Factory output = {gn:.2f} x extracted Chl-a, so true Chl-a = {ff:.2f} x the factory value:")
        print(f"  the factory calibration {direction} by ~{pct:.0f}% vs extracted Chl-a on this shelf.")
        print(f"- Roesler et al. (2017): WET Labs factory cals overestimate Chl-a by ~2x globally; a {ff:.2f}")
        print(f"  correction here sits much closer to truth than that global average.")
        print(f"- R2 {r2:.2f} (Pearson r {st['pearson_r']:.2f}): the moderate scatter is EXPECTED for a fluorescence")
        print(f"  proxy (community composition, photoacclimation, residual quenching), not a calibration failure.")
        print(f"- {st['n_npq']} near-surface daytime points are NPQ-flagged but KEPT in this variant (not removed):")
        print(f"  surface fluorescence is light-suppressed and reads low, which pulls the fit; compare with the NPQ-removed step07.")
        print(f"- Agreement with bottles improves {st['rmse_raw']:.2f} -> {st['rmse_corr']:.2f} mg/m3: usable as a")
        print(f"  QUANTITATIVE Chl-a proxy here to about +/-{st['rmse_corr']:.2f} mg/m3 below the quenched layer.")
        print(f"- Gain 95% CI {st['gain_lo']:.2f} to {st['gain_hi']:.2f} ({'excludes' if ci_excl1 else 'includes'} 1.0), n={st['n']}: "
              + ("the offset from the factory scale is statistically resolved." if ci_excl1
                 else "consistent with the factory scale within uncertainty; more deep, high-Chl bottles would sharpen it."))
        print(f"- CAVEAT: the reference is extracted FLUOROMETRIC Chl-a, not HPLC, and can run slightly high")
        print(f"  (phaeopigment / chl-b interference), so part of the near-1 factor may be the reference, not the sensor.")
    else:
        print(f"- SBE 43 oxygen vs Winkler (the reference method): sensor = {gn:.3f} x Winkler {off:+.2f} {cfg['units']}.")
        print(f"- R2 {r2:.3f}, RMSE {st['rmse_raw']:.2f} -> {st['rmse_corr']:.2f} {cfg['units']}, n={st['n']};")
        print(f"  gain 95% CI {st['gain_lo']:.3f} to {st['gain_hi']:.3f} ({'excludes' if ci_excl1 else 'includes'} 1.0).")
    print()

for _k in RESULTS: infer(_k)
for _k,_c in TARGETS.items():
    if _k not in RESULTS: print(f"{_c['label']}: no discrete file yet -> no inference.")

In [ ]:
# ------------------------------------------------------------------ apply -> corrected profiles (L3)
for key,R in RESULTS.items():
    if not np.isfinite(R["offset"]): continue
    off,gn,cfg=R["offset"],R["gain"],R["cfg"]; recs=[]
    for cid,path in CTDF.items():
        names,ddf,pcol=parse_cnv_downcast(path)
        scol=next((c for c in cfg["sensor_cols"] if c in names),None)
        if scol is None: continue
        t=ddf[[pcol,scol]].copy(); t.columns=["pressure_db","sensor_raw"]
        t["cast"]=cid; t[f"{key}_corrected"]=(t["sensor_raw"]-off)/gn
        recs.append(t)
    if recs:
        L3=pd.concat(recs,ignore_index=True)
        L3.to_csv(L3_DIR/f"{CRUISE_ID}__{key}_L3_corrected.csv",index=False,encoding="utf-8-sig")
        print(f"{cfg['label']}: wrote corrected profiles for {L3['cast'].nunique()} casts -> {L3_DIR.name}/{CRUISE_ID}__{key}_L3_corrected.csv")

In [ ]:
# ------------------------------------------------------------------ calibration scatter per active target
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
for key,R in RESULTS.items():
    if not np.isfinite(R["offset"]): continue
    cal,fit,off,gn,cfg=R["cal"],R["fit"],R["offset"],R["gain"],R["cfg"]; st=R["stats"]
    fig=Figure(figsize=(5.6,5.2)); FigureCanvasAgg(fig); ax=fig.subplots()
    for cst,gp in fit.groupby("cast"): ax.plot(gp["bottle"],gp["sensor"],"o",ms=7,label=cst)
    npq=cal[cal["npq"]&cal["sensor"].notna()]
    if len(npq): ax.plot(npq["bottle"],npq["sensor"],"x",ms=8,color="#999",label="NPQ-flagged (kept)")
    xx=np.linspace(0,max(fit["bottle"].max(),cal["bottle"].max())*1.05,50)
    ax.plot(xx,off+gn*xx,"-",color="#333",lw=1.3,label=f"fit {off:+.2f}+{gn:.3f}x")
    ax.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
    ax.set_xlabel(f"{cfg.get('discrete_label',cfg['label'])} ({cfg['units']})"); ax.set_ylabel(f"{cfg.get('sensor_label','sensor')} ({cfg['units']})")
    ax.set_title(f"{CRUISE_ID} {cfg['label']} calibration  R2={st.get('R2',np.nan):.3f} n={st.get('n',0)}",fontsize=10)
    ax.legend(fontsize=8,frameon=False)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    fig.tight_layout(); out=OUT_DIR/f"{CRUISE_ID}__{key}_calibration_scatter.png"; fig.savefig(out,dpi=135)
    try: plt.close(fig)
    except Exception: pass
    if Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: pass
    print("  saved",out.name)

In [ ]:
# ------------------------------------------------------------------ interactive: productivity explorer
try:
    import ipywidgets as W; _HAVE_W=True
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets ipympl then Restart + Run All.")

def _prod_png(key, cast_sel):
    import io
    R=RESULTS[key]; cal,fit,off,gn,cfg=R["cal"],R["fit"],R["offset"],R["gain"],R["cfg"]; st=R["stats"]
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig=Figure(figsize=(12.0,6.0)); FigureCanvasAgg(fig); ax1,ax2=fig.subplots(1,2)
    for cst,gp in fit.groupby("cast"): ax1.plot(gp["bottle"],gp["sensor"],"o",ms=7,label=cst)
    npq=cal[cal["npq"]&cal["sensor"].notna()]
    if len(npq): ax1.plot(npq["bottle"],npq["sensor"],"x",ms=8,color="#999",label="NPQ-flagged (kept)")
    if np.isfinite(off):
        xx=np.linspace(0,cal["bottle"].max()*1.05,50); ax1.plot(xx,off+gn*xx,"-",color="#333",lw=1.3,label="fit")
        ax1.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
        ax1.set_title(f"{cfg['label']}  R2={st.get('R2',np.nan):.3f}  RMSE {st.get('rmse_raw',np.nan):.2f}->{st.get('rmse_corr',np.nan):.2f}",fontsize=9)
    ax1.set_xlabel(f"{cfg.get('discrete_label','discrete')} ({cfg['units']})"); ax1.set_ylabel(f"{cfg.get('sensor_label','sensor')} ({cfg['units']})"); ax1.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax1.spines[sp].set_visible(False)
    path=CTDF.get(cast_sel)
    if path is not None and np.isfinite(off):
        names,ddf,pcol=parse_cnv_downcast(path); scol=next((c for c in cfg["sensor_cols"] if c in names),None)
        if scol:
            ax2.plot(ddf[scol],ddf[pcol],".",ms=2,color="#c9c9c9",label="sensor raw")
            ax2.plot((ddf[scol]-off)/gn,ddf[pcol],".",ms=2,color="#1a9850",label="corrected")
        bd=cal[cal["cast"]==cast_sel]
        ax2.plot(bd["bottle"],bd["depth_m"],"s",ms=8,color="#d62728",label="discrete",zorder=5)
        if (bd["npq"]).any():
            q=bd[bd["npq"]]; ax2.plot(q["bottle"],q["depth_m"],"s",ms=9,mfc="none",mec="#d62728",label="NPQ")
        _hi=max(2.0,float(np.nanmax(bd["bottle"]))*1.6) if len(bd) else 5.0   # clip raw sensor spikes out of view
        ax2.set_xlim(-0.5,_hi)
        ax2.invert_yaxis(); ax2.axvline(0,color="#bbb",lw=0.6); ax2.set_title(f"{cast_sel} profile",fontsize=9)
    ax2.set_xlabel(f"{cfg.get('product_label',cfg['label'])} ({cfg['units']})"); ax2.set_ylabel("pressure (db)"); ax2.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax2.spines[sp].set_visible(False)
    fig.suptitle(f"{CRUISE_ID} productivity calibration explorer",fontsize=11); fig.tight_layout(rect=[0,0,1,0.96])
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

_active=[k for k,R in RESULTS.items() if np.isfinite(R["offset"])]
if _HAVE_W and _active:
    wkey=W.Dropdown(options=_active,value=_active[0],description="parameter")
    _c0=sorted(RESULTS[_active[0]]["cal"]["cast"].unique().tolist())
    wcast=W.Dropdown(options=_c0,value=_c0[0],description="cast")
    def _sync(*_):
        cs=sorted(RESULTS[wkey.value]["cal"]["cast"].unique().tolist())
        wcast.options=cs; wcast.value=cs[0] if cs else None
    wkey.observe(_sync,"value")
    def _r(key,cast_sel): display(Image(data=_prod_png(key,cast_sel)))
    out=W.interactive_output(_r,{"key":wkey,"cast_sel":wcast}); display(W.VBox([W.HBox([wkey,wcast]),out]))
elif not _active:
    print("No active targets to explore (add discrete files).")